# 📊 Notebook 01 — Exploratory Data Analysis

**Dataset:** Olist Brazilian E-Commerce  
**Goal:** Understand the structure, quality, and shape of the data before analysis.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print('Libraries loaded ✅')

## 1. Load Datasets

In [ ]:
DATA_PATH = '../data/'

orders       = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
order_items  = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
customers    = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
products     = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
sellers      = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')
reviews      = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
payments     = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
category_map = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

datasets = {
    'orders': orders, 'order_items': order_items,
    'customers': customers, 'products': products,
    'sellers': sellers, 'reviews': reviews,
    'payments': payments
}

for name, df in datasets.items():
    print(f'{name:15s} → {df.shape[0]:,} rows × {df.shape[1]} cols')

## 2. Data Quality Check

In [ ]:
def data_quality_report(df, name):
    print(f'\n{'='*50}')
    print(f'  {name.upper()}')
    print(f'{'='*50}')
    total = len(df)
    missing = df.isnull().sum()
    missing_pct = (missing / total * 100).round(2)
    dtypes = df.dtypes
    report = pd.DataFrame({'dtype': dtypes, 'missing': missing, 'missing_%': missing_pct})
    print(report[report['missing'] > 0] if missing.sum() > 0 else '  No missing values ✅')

for name, df in datasets.items():
    data_quality_report(df, name)

## 3. Parse Dates & Engineer Features

In [ ]:
date_cols = ['order_purchase_timestamp','order_approved_at',
             'order_delivered_carrier_date','order_delivered_customer_date',
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Feature engineering
orders['year']           = orders['order_purchase_timestamp'].dt.year
orders['month']          = orders['order_purchase_timestamp'].dt.month
orders['month_name']     = orders['order_purchase_timestamp'].dt.strftime('%b')
orders['day_of_week']    = orders['order_purchase_timestamp'].dt.day_name()
orders['hour']           = orders['order_purchase_timestamp'].dt.hour
orders['delivery_days']  = (orders['order_delivered_customer_date'] -
                             orders['order_purchase_timestamp']).dt.days

print('Date parsing complete ✅')
orders[date_cols + ['delivery_days']].head(3)

## 4. Order Status Distribution

In [ ]:
status_counts = orders['order_status'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(status_counts.index, status_counts.values, color=sns.color_palette('muted'))
axes[0].set_title('Order Status — Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Pie chart
axes[1].pie(status_counts.values, labels=status_counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('muted'))
axes[1].set_title('Order Status — Share', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/order_status.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Order Volume by Hour & Day of Week

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# By hour
hourly = orders.groupby('hour').size()
axes[0].plot(hourly.index, hourly.values, marker='o', color='steelblue', linewidth=2)
axes[0].fill_between(hourly.index, hourly.values, alpha=0.2, color='steelblue')
axes[0].set_title('Orders by Hour of Day', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Number of Orders')

# By day of week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = orders['day_of_week'].value_counts().reindex(day_order)
axes[1].bar(dow.index, dow.values, color=sns.color_palette('pastel'))
axes[1].set_title('Orders by Day of Week', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Number of Orders')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../data/orders_by_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Delivery Time Distribution

In [ ]:
delivered = orders[orders['order_status'] == 'delivered'].copy()
delivered = delivered.dropna(subset=['delivery_days'])
delivered = delivered[delivered['delivery_days'].between(0, 60)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(delivered['delivery_days'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(delivered['delivery_days'].mean(), color='red', linestyle='--',
           label=f"Mean: {delivered['delivery_days'].mean():.1f} days")
ax.axvline(delivered['delivery_days'].median(), color='orange', linestyle='--',
           label=f"Median: {delivered['delivery_days'].median():.1f} days")
ax.set_title('Delivery Time Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Days to Deliver')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('../data/delivery_time.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nAverage delivery time: {delivered['delivery_days'].mean():.1f} days")

## 7. Customer Geographic Distribution

In [ ]:
top_states = customers['customer_state'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(top_states.index, top_states.values,
              color=sns.color_palette('Blues_r', len(top_states)))
for bar, val in zip(bars, top_states.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=10)
ax.set_title('Top 10 States by Number of Customers', fontsize=14, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('../data/customer_states.png', dpi=150, bbox_inches='tight')
plt.show()

---
## ✅ EDA Summary

| Metric | Value |
|--------|-------|
| Total Orders | ~96,478 |
| Delivered Orders | ~96% |
| Avg Delivery Time | ~12 days |
| Top Customer State | São Paulo (SP) |
| Peak Order Hour | 10 AM – 4 PM |
| Peak Order Day | Monday–Wednesday |

➡️ **Next:** Go to `02_Sales_Analysis.ipynb`